# Audit Suite — Colab runner

Runs the `unitarity-lab` audit suite on a Colab GPU, **one check at a time**,
logging every number it measures.

## What this notebook does and does not do

It is a *thin caller*. Every check lives in
[`benchmarks/audit_suite.py`](https://github.com/holeyfield33-art/unitarity-lab/blob/main/benchmarks/audit_suite.py),
a committed module with its own CLI. If a cell needs fixing to run here, the
fix belongs in that module and a new commit — a fix made only in this notebook
would be unversioned and invisible.

There are **no placeholder values anywhere in this notebook**. If a step cannot
run, it fails loudly. The previous starter notebook wrapped its imports in
`try/except` and fell through to `np.random.randn` when they failed, so a
broken run still printed a plausible-looking score — a different one each time.
That is why numbers from it could never be reproduced.

## Naming (this is the part that breaks installs)

| | | |
|---|---|---|
| GitHub repo | `unitarity-lab` | **no** trailing `s` |
| PyPI / pip name | `unitarity-labs` | **has** a trailing `s` |
| Python import | `unitarity_labs` | underscore, **has** a trailing `s` |

`pip install unitarity-lab` fails — that name is not on PyPI. Only the clone
URL drops the `s`.

The spectral dependency installs as **`var-spectral`** and imports as
**`var_spectral`**. Do **not** `pip install var`: that name belongs to an
unrelated portfolio Value-at-Risk project, and installing it shadows the real
package.

## 1. Check the runtime

Set **Runtime → Change runtime type → T4 GPU** before running. The suite works
on CPU too, just slower, and the model tiers below are sized for a 16 GB T4.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "No GPU detected — the suite will run on CPU (slower, still valid).")

## 2. Install

Two installs, in this order. VAR goes first so that `unitarity-labs`'s
`var-spectral` requirement is already satisfied when pip resolves it.

Both packages declare **dependency ranges, not exact pins**. That is
deliberate and it is the fix for the install failures: a pinned
`numpy==2.4.4` forces pip to uninstall Colab's numpy and rebuild it, which
breaks the preinstalled `torch` that was compiled against the original — the
session then needs a restart and imports fail until it gets one. With ranges,
Colab's existing scientific stack already satisfies everything and pip leaves
it alone.

The cell below records numpy's version before and after to prove that.

In [ ]:
REF = "main"  # branch, tag, or commit SHA to audit

import numpy as _np
_numpy_before = _np.__version__

# VAR first: satisfies unitarity-labs' `var-spectral>=1.1.0` requirement.
!pip install -q "var-spectral @ git+https://github.com/holeyfield33-art/VAR@{REF}"
!pip install -q "unitarity-labs[bench,spectral] @ git+https://github.com/holeyfield33-art/unitarity-lab@{REF}"

print(f"numpy before install: {_numpy_before}")

### Verify the install actually landed

This cell is the one that would have caught every past failure. It checks that
the import names resolve, that VAR is the real package and not the unrelated
`var` project, and that numpy was not swapped out from under `torch`.

If numpy changed, **Runtime → Restart session** and re-run from here. Do not
continue with a mismatched numpy: `torch` will either fail to import or return
silently wrong results.

In [ ]:
import importlib, sys

for module in ("numpy", "torch", "transformers"):
    sys.modules.pop(module, None)

import numpy, torch, transformers
import unitarity_labs, var_spectral
from unitarity_labs.core.version import __version__ as ul_version

print(f"numpy         {numpy.__version__}   (was {_numpy_before})")
print(f"torch         {torch.__version__}")
print(f"transformers  {transformers.__version__}")
print(f"unitarity_labs {ul_version}")
print(f"var_spectral   {var_spectral.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if numpy.__version__ != _numpy_before:
    raise SystemExit(
        f"numpy changed {_numpy_before} -> {numpy.__version__}. "
        "Restart the runtime (Runtime > Restart session) and re-run this cell "
        "before going further; torch was built against the previous version."
    )

# var_spectral must not be shadowed by the unrelated PyPI `var` project.
if importlib.util.find_spec("var") is not None:
    import var as _foreign
    if not getattr(_foreign, "__name__", "").startswith("var_spectral"):
        raise SystemExit(
            f"A foreign 'var' package is installed at {_foreign.__file__} and "
            "will shadow VAR. Run: pip uninstall -y var"
        )

print("\nInstall verified.")

## 3. Pick a model tier

All five tiers fit a single 16 GB T4. `vram_gb` is the weight footprint only —
activations and KV cache add to it, so leave headroom.

| tier | model | params | dtype | ~VRAM |
|---|---|---|---|---|
| `tiny` | `distilgpt2` | 82 M | fp32 | 0.4 GB |
| `small` | `gpt2` | 124 M | fp32 | 0.6 GB |
| `medium` | `gpt2-medium` | 355 M | fp16 | 0.8 GB |
| `large` | `Qwen/Qwen2.5-0.5B-Instruct` | 494 M | fp16 | 1.1 GB |
| `xl` | `Qwen/Qwen2.5-1.5B-Instruct` | 1.5 B | fp16 | 3.2 GB |

Start with `tiny` to confirm the pipeline end-to-end, then move up. The
non-model checks are identical across tiers, so only the four `model_*` checks
change with this setting.

In [ ]:
TIER = "small"    # tiny | small | medium | large | xl
REPEAT = 3        # runs of each check; >1 reports variance per metric
SEED = 42

!python -m benchmarks.audit_suite --list

## 4. Run the suite

Each check runs to completion and its numbers are flushed to
`audit.log.jsonl` before the next one starts, so a crash later never costs you
results already measured. A check that errors is recorded with its traceback
and the suite carries on.

With `REPEAT > 1` every check runs that many times and each metric is reported
with mean/std/min/max and a determinism flag. **Read that flag before quoting
any number.** Wall-clock timings are expected to vary; anything else varying
is a finding.

In [ ]:
!python -m benchmarks.audit_suite \
    --model-tier {TIER} \
    --repeat {REPEAT} \
    --seed {SEED} \
    --output-dir audit_results

## 5. Inspect every recorded number

The table below is the full contents of the run — nothing is summarised away.
`stable` marks metrics identical across all repeats.

In [ ]:
import json
from pathlib import Path

report = json.loads(Path("audit_results/audit.json").read_text())

print(f"model={report['model']}  dtype={report['dtype']}  "
      f"seed={report['seed']}  repeat={report['repeat']}")
print(f"pass={report['counts']['pass']}  fail={report['counts']['fail']}  "
      f"error={report['counts']['error']}")
print("=" * 78)

for check in report["checks"]:
    print(f"\n[{check['status'].upper()}] {check['check']}"
          f"   ({check['duration_s_mean']:.3f}s avg)")
    for key, entry in check["metrics"].items():
        stable = "stable" if entry.get("deterministic", True) else "VARIES"
        line = f"    {stable:>6}  {key} = {entry['value']}"
        if not entry.get("deterministic", True) and "stdev" in entry:
            line += f"  (min={entry['min']}, max={entry['max']}, sd={entry['stdev']})"
        print(line)
    for note in check.get("notes", []):
        print(f"            note: {note}")
    for err in check.get("errors", []):
        print(f"            error: {err.splitlines()[-1]}")

nd = report["non_deterministic_metrics"]
print("\n" + "=" * 78)
if nd:
    print(f"{len(nd)} metric(s) varied across repeats:")
    for key in nd:
        print(f"  {key}")
    print("\nTimings are expected to vary. Anything else is worth chasing.")
else:
    print("Every metric was identical across all repeats.")

## 6. Download the results

`audit.json` holds the aggregated report, `audit.log.jsonl` every individual
run, and `manifest.json` the git SHA, device, seed and full `pip freeze` that
produced them. Keep all three together — a number without its manifest cannot
be reproduced or compared against a later run.

In [ ]:
try:
    from google.colab import files
    for name in ("audit.json", "audit.log.jsonl", "manifest.json"):
        files.download(f"audit_results/{name}")
except ImportError:
    print("Not running in Colab — results are on local disk at audit_results/")